In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display,clear_output
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ==============================================================================
# Stability region of a second-order recursive LTI system
# ==============================================================================

alpha1_slider=widgets.FloatSlider(value=0.5,min=-2.2,max=2.2,step=0.05,description='α₁:')
alpha2_slider=widgets.FloatSlider(value=0.5,min=-1.1,max=1.1,step=0.05,description='α₂:')
beta0_slider=widgets.FloatSlider(value=1.0,min=-3.0,max=3.0,step=0.1,description='β₀:')
out=widgets.Output()

# Explicit colors
complex_color='#4C72B0'      # blue
distinct_color='#C49A6C'     # brown
equal_color='#800020'        # burgundy
unstable_color='#B0B0B0'     # gray

def plot_stability_region(alpha1,alpha2,beta0):
    with out:
        clear_output(wait=True)

        fig,ax=plt.subplots(figsize=(11,6.5))

        # ----------------------------------------------------------------------
        # Stability triangle and discriminant parabola
        # ----------------------------------------------------------------------

        x=np.linspace(-2.4,2.4,1000)
        upper=np.ones_like(x)
        lower=np.maximum(x-1,-x-1)
        parabola=x**2/4

        # ----------------------------------------------------------------------
        # Unstable region
        # ----------------------------------------------------------------------

        ax.fill_between(x,-1.35,lower,color=unstable_color,alpha=0.18)
        ax.fill_between(x,upper,1.55,color=unstable_color,alpha=0.18)

        # ----------------------------------------------------------------------
        # Stable region: complex conjugate poles
        # ----------------------------------------------------------------------

        mask_complex=(parabola<upper)&(parabola>lower)
        ax.fill_between(
            x,parabola,upper,
            where=mask_complex,
            color=complex_color,
            alpha=0.35
        )

        # ----------------------------------------------------------------------
        # Stable region: real and distinct poles
        # ----------------------------------------------------------------------

        mask_distinct=(parabola>lower)&(parabola<upper)
        ax.fill_between(
            x,lower,parabola,
            where=mask_distinct,
            color=distinct_color,
            alpha=0.50
        )

        # ----------------------------------------------------------------------
        # Stability boundaries
        # ----------------------------------------------------------------------

        ax.plot(x,upper,color='black',linewidth=1.2)
        ax.plot(x[x<=0],(x-1)[x<=0],color='black',linewidth=1.2)
        ax.plot(x[x>=0],(-x-1)[x>=0],color='black',linewidth=1.2)

        # ----------------------------------------------------------------------
        # Real and equal poles: discriminant = 0
        # ----------------------------------------------------------------------

        ax.plot(x,parabola,color=equal_color,linewidth=2.2)

        # ----------------------------------------------------------------------
        # Current point
        # ----------------------------------------------------------------------

        ax.scatter(
            alpha1,alpha2,
            s=140,
            facecolor='white',
            edgecolor='black',
            linewidth=2,
            zorder=10
        )

        # ----------------------------------------------------------------------
        # Pole calculation
        # ----------------------------------------------------------------------

        discriminant=alpha1**2-4*alpha2
        poles=np.roots([1,alpha1,alpha2])

        # ----------------------------------------------------------------------
        # Classification
        # ----------------------------------------------------------------------

        stable=abs(alpha2)<1 and abs(alpha1)<1+alpha2

        if discriminant>1e-12:
            pole_type="Real and distinct poles"
        elif abs(discriminant)<=1e-12:
            pole_type="Real and equal poles"
        else:
            pole_type="Complex conjugate poles"

        stability="STABLE" if stable else "UNSTABLE"

        # ----------------------------------------------------------------------
        # Equation labels
        # ----------------------------------------------------------------------

        ax.text(1.55,1.04,r'$\alpha_2=1$',fontsize=12)
        ax.text(1.55,0.42,r'$\alpha_2=\frac{\alpha_1^2}{4}$',fontsize=12,rotation=34)
        ax.text(0.75,-0.85,r'$\alpha_2=-\alpha_1-1$',fontsize=12,rotation=-45)

        # ----------------------------------------------------------------------
        # Axes
        # ----------------------------------------------------------------------

        ax.axhline(0,color='black',linewidth=1.2)
        ax.axvline(0,color='black',linewidth=1.2)

        ax.set_xlim(-2.4,2.4)
        ax.set_ylim(-1.35,1.55)

        ax.set_xlabel(r'$\alpha_1$',fontsize=14)
        ax.set_ylabel(r'$\alpha_2$',fontsize=14,rotation=0,labelpad=15)

        ax.set_xticks([-2,-1,0,1,2])
        ax.set_yticks([-1,0,1])
        ax.grid(False)

        # ----------------------------------------------------------------------
        # Title
        # ----------------------------------------------------------------------

        ax.set_title(
            f'Stability Region of a Second-Order Recursive LTI System\n'
            f'α₁ = {alpha1:.2f},  α₂ = {alpha2:.2f},  β₀ = {beta0:.2f}  →  {stability}',
            fontsize=12
        )

        # ----------------------------------------------------------------------
        # Explicit legend
        # ----------------------------------------------------------------------

        legend_elements=[
            Patch(
                facecolor=complex_color,
                alpha=0.35,
                label='Stable – complex conjugate poles'
            ),
            Patch(
                facecolor=distinct_color,
                alpha=0.50,
                label='Stable – real and distinct poles'
            ),
            Line2D(
                [0],[0],
                color=equal_color,
                linewidth=3,
                label='Real and equal poles'
            ),
            Patch(
                facecolor=unstable_color,
                alpha=0.40,
                label='Unstable region'
            )
        ]

        ax.legend(
            handles=legend_elements,
            loc='center left',
            bbox_to_anchor=(1.02,0.5),
            frameon=True,
            fontsize=10,
            title='Pole / stability classification',
            title_fontsize=11
        )

        # ----------------------------------------------------------------------
        # Layout
        # ----------------------------------------------------------------------

        fig.subplots_adjust(right=0.73)
        plt.show()

        # ----------------------------------------------------------------------
        # Numerical information
        # ----------------------------------------------------------------------

        print(f"Pole classification: {pole_type}")
        print(f"p₁ = {poles[0]:.5f}")
        print(f"p₂ = {poles[1]:.5f}")
        print(f"|p₁| = {abs(poles[0]):.5f}")
        print(f"|p₂| = {abs(poles[1]):.5f}")

def update(alpha1,alpha2,beta0):
    plot_stability_region(alpha1,alpha2,beta0)

display(alpha1_slider,alpha2_slider,beta0_slider)
display(out)

widgets.interactive_output(
    update,
    {'alpha1':alpha1_slider,'alpha2':alpha2_slider,'beta0':beta0_slider}
)

plot_stability_region(
    alpha1_slider.value,
    alpha2_slider.value,
    beta0_slider.value
)